# Chapter 11: Payment System
*System Design Interview Volume 2*

## TL;DR

A payment system backend for e-commerce separates **pay-in** (buyer to platform via PSP) and **pay-out** (platform to seller). The payment service orchestrates events through a payment executor, PSP, ledger, and wallet. Key engineering challenges: **idempotency** (prevent double charges), **reconciliation** (detect inconsistencies with external systems), and **retry with dead letter queues** (handle transient failures gracefully). A double-entry ledger ensures every cent is traceable. Security relies on hosted payment pages (avoid PCI DSS burden), tokenization, and HTTPS.

## Requirements

| Requirement | Detail |
|---|---|
| Pay-in flow | Receive money from buyers on behalf of sellers via credit card / PSP |
| Pay-out flow | Send money to sellers around the world via third-party providers |
| Reliability | Handle failed payments; no double charges |
| Reconciliation | Async verification between internal services and external PSPs/banks |
| Scale | 1 million transactions/day (~10 TPS) |
| Security | PCI DSS compliance; no raw card data stored internally |

## Estimation

In [ ]:
# Back-of-the-envelope: Payment System
transactions_per_day = 1_000_000
seconds_per_day = 86_400

tps = transactions_per_day / seconds_per_day
print(f"Average TPS: {tps:.1f}")
print(f"Peak TPS (10x): {tps * 10:.0f}")
print()

# Payment event storage (1 year)
avg_event_bytes = 500        # checkout_id, buyer_info, card_token, status
events_per_year = transactions_per_day * 365
storage_gb = events_per_year * avg_event_bytes / (1024**3)
print(f"Payment events/year: {events_per_year:,.0f}")
print(f"Storage/year: {storage_gb:.1f} GB")
print()

# Payment orders (avg 2 orders per event)
orders_per_event = 2
orders_per_year = events_per_year * orders_per_event
order_bytes = 300
order_storage_gb = orders_per_year * order_bytes / (1024**3)
print(f"Payment orders/year: {orders_per_year:,.0f}")
print(f"Order storage/year: {order_storage_gb:.1f} GB")

## High-Level Design: Pay-in Flow

```mermaid
graph LR
    Client["Client Browser/App"]
    PS["Payment Service"]
    PE["Payment Executor"]
    PSP["PSP (Stripe/Adyen)"]
    CS["Card Schemes (Visa/MC)"]
    L["Ledger"]
    W["Wallet"]
    DB[("Payment DB")]

    Client -->|"1. Payment event"| PS
    PS -->|"2. Store event"| DB
    PS -->|"3. Payment order"| PE
    PE -->|"4. Store order"| DB
    PE -->|"5. Process payment"| PSP
    PSP --> CS
    PS -->|"6. Update balance"| W
    PS -->|"7. Update ledger"| L

    style PS fill:#b3e6b3,stroke:#333
    style PE fill:#ffcc99,stroke:#333
    style PSP fill:#ff9999,stroke:#333
```

**Flow:** User clicks pay --> Payment service stores event --> Payment executor sends order to PSP --> PSP processes via card scheme --> On success, wallet and ledger are updated.

**Key:** A single payment *event* (checkout) may contain multiple payment *orders* (one per seller).

## Deep Dive: PSP Integration via Hosted Payment Page

```mermaid
sequenceDiagram
    participant B as Browser
    participant PS as Payment Service
    participant PSP as PSP (Stripe)

    B->>PS: 1. Checkout with order info
    PS->>PSP: 2. Register payment (nonce = order UUID)
    PSP-->>PS: 3. Return token
    PS->>PS: 4. Store token in DB
    B->>PSP: 5. Hosted page collects card details
    PSP->>PSP: 6. Process payment
    PSP-->>B: 7. Payment status
    B->>B: 8. Redirect to success/failure page
    PSP->>PS: 9. Webhook with completion result
```

**Why hosted pages?** The PSP collects card data directly -- our system **never touches raw card numbers**, avoiding PCI DSS compliance burden. The nonce (payment_order_id) serves as the PSP-side idempotency key.

## Deep Dive: Reconciliation

```mermaid
graph TB
    subgraph internal["Internal Systems"]
        PS2["Payment Service"]
        L2["Ledger"]
        W2["Wallet"]
    end

    subgraph external["External Systems"]
        PSP2["PSP"]
        Bank["Bank"]
    end

    SF["Settlement File (nightly)"]
    REC["Reconciliation Service"]

    PSP2 -->|"Nightly"| SF
    Bank -->|"Nightly"| SF
    SF --> REC
    L2 --> REC
    REC -->|"Auto-fix"| AF["Automated Adjustment"]
    REC -->|"Manual fix"| MQ["Finance Team Queue"]
    REC -->|"Unknown"| IQ["Investigation Queue"]

    style REC fill:#ffcc99,stroke:#333
```

**Three mismatch categories:**
1. **Classifiable + automatable** -- known cause, cost-effective to code a fix
2. **Classifiable + manual** -- known cause, but automated fix too expensive
3. **Unclassifiable** -- unknown cause, requires manual investigation

## Deep Dive: Retry & Idempotency (Exactly-Once)

```mermaid
graph TB
    Pay["Payment Request"]
    Sys["Payment System"]
    RQ["Retry Queue"]
    DLQ["Dead Letter Queue"]
    DB2[("Database")]

    Pay --> Sys
    Sys -->|"Success"| DB2
    Sys -->|"Retryable failure"| RQ
    RQ -->|"Retry"| Sys
    RQ -->|"Exceeds threshold"| DLQ
    Sys -->|"Non-retryable failure"| DB2

    style RQ fill:#ffcc99,stroke:#333
    style DLQ fill:#ff9999,stroke:#333
```

**Exactly-once = at-least-once (retry) + at-most-once (idempotency)**

| Retry Strategy | Description |
|---|---|
| Immediate | Resend instantly |
| Fixed interval | Wait constant time between retries |
| Incremental | Increase wait linearly |
| Exponential backoff | Double wait each retry (1s, 2s, 4s, 8s...) |
| Cancel | Abort on permanent failure |

**Idempotency mechanism:** The  is the primary key (and idempotency key). INSERT succeeds only once; duplicates hit a unique constraint violation and return the cached result. The PSP uses the token as its own idempotency key.

## Deep Dive: Double-Entry Ledger

Every transaction produces two entries that sum to zero:

| Account | Debit | Credit |
|---|---|---|
| Buyer |  | |
| Seller | |  |
| **Sum** | **** | **** |

**Why strings for amounts?** Different protocols handle numeric precision differently during serialization. Floating-point can lose precision with very large (Japan GDP ~ 5x10^14 yen) or very small (Bitcoin satoshi = 10^-8) values. Store as strings; parse only for display/calculation.

## Deep Dive: Async Communication

```mermaid
graph LR
    subgraph single["Single Receiver (Queue)"]
        M1["m1"] --> SA["Service A"]
        M2["m2"] --> SB["Service B"]
    end

    subgraph multi["Multiple Receivers (Kafka)"]
        K["Kafka Topic"]
        K --> PS3["Payment System"]
        K --> AN["Analytics"]
        K --> BL["Billing"]
    end
```

**Kafka fan-out** is preferred for large-scale payment systems: one payment event triggers updates to payment processing, analytics, billing, and push notifications simultaneously.

## Trade-offs

| Decision | Pro | Con |
|---|---|---|
| Hosted payment page over direct API | No PCI DSS burden; PSP handles security | Less UI control; redirect latency |
| Async over sync communication | Scalable, fault-isolated, decoupled | Eventual consistency; harder to debug |
| Exponential backoff retry | Prevents overload; self-healing | Slow recovery; needs max retry cap |
| Idempotency via DB primary key | Simple, strong guarantee | Requires globally unique order IDs |
| Double-entry ledger | Full traceability; sum-to-zero invariant | More storage; more complex writes |
| Nightly reconciliation | Catches all drift; last line of defense | Up to 24h delay in detecting issues |

## Takeaways

1. **Separate pay-in and pay-out** -- different flows, different third-party providers, different timing
2. **Never store raw card data** -- use hosted payment pages to offload PCI DSS compliance to the PSP
3. **Idempotency is non-negotiable** -- use payment_order_id as the idempotency key everywhere
4. **Reconciliation is the last line of defense** -- nightly settlement file comparison catches what retries miss
5. **Double-entry ledger** ensures every cent is accounted for and traceable
6. **Prefer async (Kafka)** for internal communication -- enables fan-out and failure isolation
7. **10 TPS is low** -- the design challenge is correctness, not throughput

## Related Concepts

- [[payment-system-architecture]] -- Pay-in/pay-out flow and component roles
- [[psp-integration]] -- Hosted payment page and token-based flow
- [[reconciliation]] -- Settlement file comparison and mismatch handling
- [[retry-and-idempotency]] -- Exactly-once delivery via retry + idempotency
- [[double-entry-ledger]] -- Accounting principle for payment traceability
- [[payment-consistency]] -- Distributed consistency techniques
- [[payment-security]] -- Tokenization, PCI DSS, and fraud prevention